# Downloading WorldStrat dataset

This notebook was adapted from WorldStrat.

[Open the main dataset repository on Zenodo](https://zenodo.org/record/6810792), and download:
- [The high-resolution imagery](https://zenodo.org/record/6810792/files/hr_dataset.tar.gz?download=1).
- [The low-resolution imagery (Sentinel-2 Level-2A)](https://zenodo.org/record/6810792/files/lr_dataset_l2a.tar.gz?download=1) for each high-resolution image.
- [The metadata](https://zenodo.org/record/6810792/files/metadata.csv?download=1) (imaged locations coordinates, several classifications).
- [The train/val/test split](https://zenodo.org/record/6810792/files/stratified_train_val_test_split.csv?download=1) used to train the super-resolution benchmarks.

**The cell below can download the dataset for you, just set the `DOWNLOAD` variable to `True`:**  
- If the download gets interrupted, you can just rerun the cell and it will just continue downloading from where it was interrupted. 
- The download location is controlled by the `DATA_ROOT` environment variable (loaded from the `.env` file at the repository root); files are stored in `$DATA_ROOT/satellite_dataset`.
- If the data is already downloaded in `$DATA_ROOT/satellite_dataset`, it will skip the download.
- NOTE: **This will take a while.**

In [ ]:
########################################################
# SET THE VARIABLE BELOW TO TRUE TO DOWNLOAD THE DATASET
DOWNLOAD = True
########################################################
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join('..', '.env'))

DATA_ROOT = os.getenv('DATA_ROOT')

from dataset_download_tools import *
files_to_download = {
    os.path.join(DATA_ROOT, 'satellite_dataset', 'hr_dataset.tar.gz') : 'https://zenodo.org/record/6810792/files/hr_dataset.tar.gz?download=1',
    os.path.join(DATA_ROOT, 'satellite_dataset', 'lr_dataset_l2a.tar.gz') : 'https://zenodo.org/record/6810792/files/lr_dataset_l2a.tar.gz?download=1',
    os.path.join(DATA_ROOT, 'satellite_dataset', 'metadata.csv') : 'https://zenodo.org/record/6810792/files/metadata.csv?download=1',
    os.path.join(DATA_ROOT, 'satellite_dataset', 'stratified_train_val_test_split.csv') : 'https://zenodo.org/record/6810792/files/stratified_train_val_test_split.csv?download=1',
}
if DOWNLOAD:
    for file, link in files_to_download.items():
        download_large_file(link, file)

**The files are expected in `$DATA_ROOT/satellite_dataset` (where `DATA_ROOT` is set in the `.env` file at the repository root). The directory used for verification is set in the cell below; update `directory` there if you downloaded the files elsewhere.**

If the files were downloaded using the cell above, just run the cell below.


In [ ]:
######################################################
# ENTER YOUR DOWNLOAD DIRECTORY BELOW
# If you used the cell above to download the dataset
# You can just run the cell as-is.
directory = os.path.join(DATA_ROOT, 'satellite_dataset')
######################################################

from dataset_download_tools import *
expected_dataset_hashes = {
    'hr_dataset.tar.gz' :   'ca7167334006f3c17f9071f14c435335',
    'lr_dataset_l2a.tar.gz':'8cfc6a477cee9e9cd8b20ea27227de65',
    'metadata.csv':         'dfeb3348e79b719bf03c230d5d258839',
    'stratified_train_val_test_split.csv':'745035835d835280aa0298a9dc1996d1'
}

result = check_md5_hash(directory, expected_dataset_hashes)

In [ ]:
if result:
    print('✅ Hashes verified. All good.')
else:
    print('❌ Please check downloaded files. Hashes do not match.')

The dataset is extracted into `$DATA_ROOT/satellite_dataset` (where `DATA_ROOT` is set in the `.env` file at the repository root); this is the `directory` used in the cell below.

You should now have the following structure within `$DATA_ROOT/satellite_dataset`:
- `hr_dataset\{AOI NAME}\`
- `lr_dataset\{AOI NAME}\L1C`
- `lr_dataset\{AOI NAME}\L2A`
- `stratified_train_val_test_split.csv`
- `metadata.csv`

**The cell below can extract the dataset for you, just set the `EXTRACT` variable to `True`:**  
- If the extraction gets interrupted, you can just rerun the cell and it will just continue extracting from where it was interrupted.
- NOTE: **This will take a while.**

In [ ]:
import os
import tarfile

#############################################################################
# SET THE VARIABLE BELOW TO TRUE TO EXTRACT THE DATASET
EXTRACT = True
# IF YOU DOWNLOADED THE DATASET MANUALLY INPUT YOUR DOWNLOAD DIRECTORY BELOW
directory = os.path.join(DATA_ROOT, 'satellite_dataset')
#############################################################################

def extract_tar_gz(archive_path, output_folder, num_files, filter_func=None):
    with tarfile.open(archive_path, 'r:gz') as tar:
        members = tar.getmembers()
        if filter_func is not None:
            members = [m for m in members if filter_func(m)]
        tar.extractall(path=output_folder, members=members)

def filter_l2a_data_tiff(tarinfo):
    return "L2A_data.tiff" in tarinfo.name

def filter_ps_tiff(tarinfo):
    return "ps.tiff" in tarinfo.name

if EXTRACT:
    extract_tar_gz(os.path.join(directory,'lr_dataset_l2a.tar.gz'), directory, 573338, filter_func=filter_l2a_data_tiff)
    extract_tar_gz(os.path.join(directory, 'hr_dataset.tar.gz') , directory, 19645, filter_func=filter_ps_tiff)

The following cell can be used to spot if there are missing Sentinel L2A images in WorldStrat. When implementing this code we were missing images from POIs 'ASMSpotter-1-1-1', 'Landcover-1062091', 'UNHCR-HUNs002665'. These POIs are then excluded in LoadDataset.py.

In [ ]:
import os
import tarfile
import pandas as pd
from tqdm import tqdm

Verify = True
download_directory = os.path.join(DATA_ROOT, 'satellite_dataset')

def get_filenames_from_tar(archive_path):
    with tarfile.open(archive_path, 'r:gz') as tar:
        members = tar.getmembers()
        filenames = {member.name.split('/')[-1].split('-L2A_data.tiff')[0] for member in tqdm(members, desc="Reading tar contents", unit="file") if '-L2A_data.tiff' in member.name}
    return filenames

def verify_poi_in_filenames(metadata_path, filenames):
    # Read metadata
    data = pd.read_csv(metadata_path)
    data = data.rename(columns={data.columns[0]: 'POI'})
    data['image'] = data['POI'] + '-' + data['n'].astype(str)
    
    # Get unique POIs
    images = set(data['image'].tolist())
    
    # Check for missing POIs
    missing_pois = [image for image in tqdm(images, desc="Verifying POIs", unit="image") if image not in filenames]
    
    return missing_pois

if Verify:
    # Paths to the relevant files
    archive_path = os.path.join(download_directory, 'lr_dataset_l2a.tar.gz')
    metadata_path = os.path.join(download_directory, 'metadata.csv')

    # Get filenames from the tar archive
    filenames = get_filenames_from_tar(archive_path)

    # Verify POIs in filenames
    missing_images = verify_poi_in_filenames(metadata_path, filenames)

    # Print or handle the missing POIs as needed
    if missing_images:
        print(f"Missing images: {len(missing_images)}")
        print("First 10 missing images:", missing_images[:10])
    else:
        print("All POIs are present in the filenames.")